In [36]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("HomeCredit_Serving_Layer") \
    .enableHiveSupport() \
    .getOrCreate()

In [37]:
raw_app = spark.read.parquet("/user/student/home_credit/raw/application_train")
raw_prev = spark.read.parquet("/user/student/home_credit/raw/previous_application")
raw_inst = spark.read.parquet("/user/student/home_credit/raw/installments_payments")
raw_bureau = spark.read.parquet("/user/student/home_credit/raw/bureau")

print("Raw Parquet Data loaded successfully from HDFS!")

Raw Parquet Data loaded successfully from HDFS!


In [38]:
# 1. Application Train
df_app = raw_app.select(
    F.col("SK_ID_CURR").cast("int"),
    F.col("TARGET").cast("int"),
    F.col("NAME_CONTRACT_TYPE").cast("string"),
    F.col("DAYS_BIRTH").cast("int"),
    F.col("OCCUPATION_TYPE").cast("string"),
    F.col("NAME_EDUCATION_TYPE").cast("string"),
    F.col("NAME_FAMILY_STATUS").cast("string"),
    F.col("NAME_HOUSING_TYPE").cast("string"),
    F.col("NAME_INCOME_TYPE").cast("string"),
    F.col("FLAG_OWN_REALTY").cast("string"),
    F.col("AMT_INCOME_TOTAL").cast("double"),
    F.col("AMT_CREDIT").cast("double"),
    F.col("AMT_ANNUITY").cast("double"),
    F.col("AMT_GOODS_PRICE").cast("double"),
    F.col("DAYS_EMPLOYED").cast("int")
)

# 2. Previous Applications
df_prev = raw_prev.select(
    F.col("SK_ID_PREV").cast("int"),
    F.col("SK_ID_CURR").cast("int"),
    F.col("NAME_CONTRACT_STATUS").cast("string"),
    F.col("AMT_CREDIT").cast("double")
)

# 3. Installments Payments
df_inst = raw_inst.select(
    F.col("SK_ID_PREV").cast("int"),
    F.col("SK_ID_CURR").cast("int"),
    F.col("DAYS_INSTALMENT").cast("double"),
    F.col("DAYS_ENTRY_PAYMENT").cast("double"),
    F.col("AMT_INSTALMENT").cast("double"),
    F.col("AMT_PAYMENT").cast("double")
)

# 4. Bureau Data
df_bureau = raw_bureau.select(
    F.col("SK_ID_CURR").cast("int"),
    F.col("CREDIT_ACTIVE").cast("string"),
    F.col("AMT_CREDIT_SUM_DEBT").cast("double"),
    F.col("AMT_CREDIT_SUM_OVERDUE").cast("double"),
    F.col("AMT_CREDIT_MAX_OVERDUE").cast("double"),
    F.col("AMT_CREDIT_SUM").cast("double")
)

print("Data selected successfully!")

Data selected successfully!


In [39]:
from pyspark.sql.types import StringType
from pyspark.sql import functions as F

def clean_empty_strings(df):
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    
    for c in string_cols:
        df = df.withColumn(c, F.when(F.trim(F.col(c)) == "", None).otherwise(F.col(c)))
        
    return df

df_app = clean_empty_strings(df_app)
df_prev = clean_empty_strings(df_prev)
df_inst = clean_empty_strings(df_inst)
df_bureau = clean_empty_strings(df_bureau)

print("Transformations applied")

Transformations applied


In [40]:
clean_app = df_app.dropDuplicates(["SK_ID_CURR"]).filter(F.col("SK_ID_CURR").isNotNull())
clean_prev = df_prev.dropDuplicates(["SK_ID_PREV"]).filter(F.col("SK_ID_CURR").isNotNull())
clean_inst = df_inst.dropDuplicates().filter(F.col("SK_ID_CURR").isNotNull())
clean_bureau = df_bureau.dropDuplicates().filter(F.col("SK_ID_CURR").isNotNull())

print("Data cleaning completed successfully!")

Data cleaning completed successfully!


In [46]:
#clean_app.printSchema()
#clean_prev.printSchema()
#clean_inst.printSchema()
#clean_bureau.printSchema()

#clean_app.show(20)
#clean_prev.show(20)
#clean_inst.show(20)
#clean_bureau.show(20)

In [42]:
clean_app_fixed = clean_app \
    .withColumn("DAYS_EMPLOYED", F.when(F.col("DAYS_EMPLOYED") == 365243, None).otherwise(F.col("DAYS_EMPLOYED"))) \
    .withColumn("OCCUPATION_TYPE", F.when(F.col("NAME_INCOME_TYPE") == "Pensioner", "Retired").otherwise(F.col("OCCUPATION_TYPE"))) \
    .withColumn("DAYS_BIRTH", F.abs(F.col("DAYS_BIRTH"))) \
    .withColumn("DAYS_EMPLOYED", F.abs(F.col("DAYS_EMPLOYED")))

In [43]:
clean_inst_fixed = clean_inst \
    .withColumn("DAYS_INSTALMENT", F.abs(F.col("DAYS_INSTALMENT"))) \
    .withColumn("DAYS_ENTRY_PAYMENT", F.abs(F.col("DAYS_ENTRY_PAYMENT")))

In [44]:
clean_bureau_fixed = clean_bureau \
    .withColumn("AMT_CREDIT_SUM", F.when(F.col("AMT_CREDIT_SUM") <= 0.0, None).otherwise(F.col("AMT_CREDIT_SUM"))) \
    .fillna(0.0, subset=["AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_OVERDUE", "AMT_CREDIT_MAX_OVERDUE"])

In [45]:
clean_app.write.mode("overwrite").parquet("/user/student/home_credit/staging/application_train")
clean_prev.write.mode("overwrite").parquet("/user/student/home_credit/staging/previous_application")
clean_inst.write.mode("overwrite").parquet("/user/student/home_credit/staging/installments_payments")
clean_bureau.write.mode("overwrite").parquet("/user/student/home_credit/staging/bureau")
print("all done")

all done
